In [2]:
from functions import *
import networkx as nx
import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt
from tqdm import tqdm

In [4]:
# =============================================================================
# 4-VERTEX CYCLE GRAPH — MANUAL VERIFICATION
# Vertices: 0, 1, 2, 3
# Edges: (0,1), (1,2), (2,3), (3,0)
# =============================================================================


In [6]:
# --- Phase 1: Build A, D, L ---
n_vertices = 4
edges = [(0,1), (1,2), (2,3), (3,0)]

A = np.zeros((n_vertices, n_vertices))
for (i, j) in edges:
    A[i, j] = 1
    A[j, i] = 1

degrees = A.sum(axis=1)
D = np.diag(degrees)
L = D - A

print("=== Phase 1 ===")
print(f"A:\n{A}")
print(f"D diagonal: {np.diag(D)}")
print(f"L:\n{L}")
print(f"L symmetric: {np.allclose(L, L.T)}")
print(f"L * 1 = {L @ np.ones(n_vertices)} (should be all zeros)")
print()

=== Phase 1 ===
A:
[[0. 1. 0. 1.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [1. 0. 1. 0.]]
D diagonal: [2. 2. 2. 2.]
L:
[[ 2. -1.  0. -1.]
 [-1.  2. -1.  0.]
 [ 0. -1.  2. -1.]
 [-1.  0. -1.  2.]]
L symmetric: True
L * 1 = [0. 0. 0. 0.] (should be all zeros)



In [8]:
from scipy.linalg import eigh
eigenvalues, _ = eigh(L, D)
lambda_min = eigenvalues[eigenvalues > 1e-10][0]
sigma_squared = 0.25
L_sigma = L + sigma_squared * lambda_min * D

print("=== Phase 2 ===")
print(f"Eigenvalues of D^-1 L: {eigenvalues}")
print(f"lambda_min = {lambda_min:.4f}")
print(f"L_sigma:\n{np.round(L_sigma, 4)}")
print(f"Smallest eigenvalue of L_sigma: {np.linalg.eigvalsh(L_sigma)[0]:.4f} (should be > 0)")
print()


=== Phase 2 ===
Eigenvalues of D^-1 L: [-1.64198576e-16  1.00000000e+00  1.00000000e+00  2.00000000e+00]
lambda_min = 1.0000
L_sigma:
[[ 2.5 -1.   0.  -1. ]
 [-1.   2.5 -1.   0. ]
 [ 0.  -1.   2.5 -1. ]
 [-1.   0.  -1.   2.5]]
Smallest eigenvalue of L_sigma: 0.5000 (should be > 0)



In [10]:
# --- Phase 3: White Noise Sampling ---
# use exact w values from our manual example
w = np.array([0.5, -0.3, 0.8, -0.2])  # w_(0,1), w_(1,2), w_(2,3), w_(3,0)

# 3b: build incidence matrix and compute f
B = np.zeros((n_vertices, len(edges)))
for idx, (i, j) in enumerate(edges):
    B[i, idx] =  1
    B[j, idx] = -1

f = B @ w

print("=== Phase 3 ===")
print(f"w (edge noise): {w}")
print(f"f (vertex sums): {f}  (expected: [0.3, 0.2, 0.5, 0.6])")

# 3c: solve L_sigma u = lambda_min * f
cho_cache = cho_factor(L_sigma)
u = cho_solve(cho_cache, lambda_min * f)

print(f"u (correlated field): {np.round(u, 4)}  (expected: [0.7601, 0.7201, 0.8400, 0.8801])")
print()


=== Phase 3 ===
w (edge noise): [ 0.5 -0.3  0.8 -0.2]
f (vertex sums): [ 0.7 -0.8  1.1 -1. ]  (expected: [0.3, 0.2, 0.5, 0.6])
u (correlated field): [ 0.12 -0.16  0.28 -0.24]  (expected: [0.7601, 0.7201, 0.8400, 0.8801])



In [12]:
# compute f directly without signed incidence matrix
f = np.zeros(n_vertices)
for idx, (i, j) in enumerate(edges):
    f[i] += w[idx]
    f[j] += w[idx]  # both vertices get the same w_e added

In [14]:
# --- Phase 3: White Noise Sampling ---
# use exact w values from our manual example
w = np.array([0.5, -0.3, 0.8, -0.2])  # w_(0,1), w_(1,2), w_(2,3), w_(3,0)

# 3b: build incidence matrix and compute f
B = np.zeros((n_vertices, len(edges)))
for idx, (i, j) in enumerate(edges):
    B[i, idx] =  1
    B[j, idx] = -1

f = B @ w

print("=== Phase 3 ===")
print(f"w (edge noise): {w}")
print(f"f (vertex sums): {f}  (expected: [0.3, 0.2, 0.5, 0.6])")

# 3c: solve L_sigma u = lambda_min * f
cho_cache = cho_factor(L_sigma)
u = cho_solve(cho_cache, lambda_min * f)

print(f"u (correlated field): {np.round(u, 4)}  (expected: [0.7601, 0.7201, 0.8400, 0.8801])")
print()


=== Phase 3 ===
w (edge noise): [ 0.5 -0.3  0.8 -0.2]
f (vertex sums): [ 0.7 -0.8  1.1 -1. ]  (expected: [0.3, 0.2, 0.5, 0.6])
u (correlated field): [ 0.12 -0.16  0.28 -0.24]  (expected: [0.7601, 0.7201, 0.8400, 0.8801])



In [16]:
# 3b: compute f directly — sum w_e for all incident edges
B = np.zeros((n_vertices, len(edges)))
for idx, (i, j) in enumerate(edges):
    B[i, idx] =  1
    B[j, idx] = -1

f = np.zeros(n_vertices)
for idx, (i, j) in enumerate(edges):
    f[i] += w[idx]
    f[j] += w[idx]

In [18]:
f = B @ w

print("=== Phase 3 ===")
print(f"w (edge noise): {w}")
print(f"f (vertex sums): {f}  (expected: [0.3, 0.2, 0.5, 0.6])")

# 3c: solve L_sigma u = lambda_min * f
cho_cache = cho_factor(L_sigma)
u = cho_solve(cho_cache, lambda_min * f)

print(f"u (correlated field): {np.round(u, 4)}  (expected: [0.7601, 0.7201, 0.8400, 0.8801])")
print()

# --- Phase 4: Compute Permeability ---
k = {}
for (i, j) in edges:
    k[(i,j)] = np.exp((u[i] + u[j]) / 2)

print("=== Phase 4 ===")
for (i,j), k_e in k.items():
    print(f"  k_({i},{j}) = {k_e:.4f}")
print(f"  (expected: 2.0961, 2.1816, 2.3633, 2.2707)")
print()

# --- Phase 5: Build L_k and solve Darcy flow ---
L_k = lil_matrix((n_vertices, n_vertices))
for (i, j) in edges:
    k_e = k[(i,j)]
    L_k[i,i] += k_e
    L_k[j,j] += k_e
    L_k[i,j] -= k_e
    L_k[j,i] -= k_e
L_k = L_k.tocsr()

print("=== Phase 5 ===")
print(f"L_k:\n{np.round(L_k.toarray(), 4)}")

# boundary conditions
gamma_in  = [0]
gamma_out = [2]
p = np.zeros(n_vertices)
p[0] = 1.0  # inlet
p[2] = 0.0  # outlet

boundary = set(gamma_in) | set(gamma_out)
interior = np.array([v for v in range(n_vertices) if v not in boundary])

L_interior = L_k[interior, :][:, interior]
rhs        = -L_k[interior, :][:, list(boundary)] @ p[list(boundary)]
p_interior = spsolve(L_interior, rhs)
p[interior] = p_interior

print(f"Pressure field p: {np.round(p, 4)}")
print(f"  (expected: p_0=1.0, p_1=0.4900, p_2=0.0, p_3=0.4899)")
print()

# --- Phase 6: Extract QoI ---
gamma_out_set = set(gamma_out)
Q = 0.0
for (i, j) in edges:
    k_e = k.get((i,j), k.get((j,i), 1.0))
    if i in gamma_out_set or j in gamma_out_set:
        Q += k_e * abs(p[i] - p[j])

print("=== Phase 6 ===")
print(f"Q = {Q:.4f}  (expected: 2.2267)")

# verify conservation: flow in = flow out
Q_in = 0.0
gamma_in_set = set(gamma_in)
for (i, j) in edges:
    k_e = k.get((i,j), k.get((j,i), 1.0))
    if i in gamma_in_set or j in gamma_in_set:
        Q_in += k_e * abs(p[i] - p[j])

print(f"Flow in  = {Q_in:.4f}")
print(f"Flow out = {Q:.4f}")
print(f"Conservation holds: {np.isclose(Q_in, Q, atol=1e-4)}")

=== Phase 3 ===
w (edge noise): [ 0.5 -0.3  0.8 -0.2]
f (vertex sums): [ 0.7 -0.8  1.1 -1. ]  (expected: [0.3, 0.2, 0.5, 0.6])
u (correlated field): [ 0.12 -0.16  0.28 -0.24]  (expected: [0.7601, 0.7201, 0.8400, 0.8801])

=== Phase 4 ===
  k_(0,1) = 0.9802
  k_(1,2) = 1.0618
  k_(2,3) = 1.0202
  k_(3,0) = 0.9418
  (expected: 2.0961, 2.1816, 2.3633, 2.2707)

=== Phase 5 ===
L_k:
[[ 1.922  -0.9802  0.     -0.9418]
 [-0.9802  2.042  -1.0618  0.    ]
 [ 0.     -1.0618  2.082  -1.0202]
 [-0.9418  0.     -1.0202  1.962 ]]
Pressure field p: [1.   0.48 0.   0.48]
  (expected: p_0=1.0, p_1=0.4900, p_2=0.0, p_3=0.4899)

=== Phase 6 ===
Q = 0.9994  (expected: 2.2267)
Flow in  = 0.9994
Flow out = 0.9994
Conservation holds: True


In [20]:
import numpy as np
from scipy.linalg import eigh, cho_factor, cho_solve
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

# =============================================================================
# 4-VERTEX CYCLE GRAPH — MANUAL VERIFICATION
# Vertices: 0, 1, 2, 3
# Edges: (0,1), (1,2), (2,3), (3,0)
# =============================================================================

n_vertices = 4
edges = [(0,1), (1,2), (2,3), (3,0)]

# --- Phase 1 ---
A = np.zeros((n_vertices, n_vertices))
for (i, j) in edges:
    A[i, j] = 1
    A[j, i] = 1

degrees = A.sum(axis=1)
D       = np.diag(degrees)
L       = D - A

print("=== Phase 1 ===")
print(f"A:\n{A}")
print(f"D diagonal: {np.diag(D)}")
print(f"L:\n{L}")
print(f"L symmetric: {np.allclose(L, L.T)}")
print(f"L * 1 = {L @ np.ones(n_vertices)} (should be all zeros)")
print()

# --- Phase 2 ---
eigenvalues, _ = eigh(L, D)
lambda_min     = eigenvalues[eigenvalues > 1e-10][0]
sigma_squared  = 0.25
L_sigma        = L + sigma_squared * lambda_min * D

print("=== Phase 2 ===")
print(f"Eigenvalues of D^-1 L: {np.round(eigenvalues, 4)}")
print(f"lambda_min = {lambda_min:.4f}")
print(f"L_sigma:\n{np.round(L_sigma, 4)}")
print(f"Smallest eigenvalue of L_sigma: {np.linalg.eigvalsh(L_sigma)[0]:.4f} (should be > 0)")
print()

# --- Phase 3 ---
# fixed w values from manual example
w = np.array([0.5, -0.3, 0.8, -0.2])  # w_(0,1), w_(1,2), w_(2,3), w_(3,0)

# 3b: compute f by summing w_e for all incident edges (no signs)
f = np.zeros(n_vertices)
for idx, (i, j) in enumerate(edges):
    f[i] += w[idx]
    f[j] += w[idx]

# 3c: solve L_sigma u = lambda_min * f
cho_cache = cho_factor(L_sigma)
u         = cho_solve(cho_cache, lambda_min * f)

print("=== Phase 3 ===")
print(f"w (edge noise)      : {w}")
print(f"f (vertex sums)     : {np.round(f, 4)}  (expected: [0.3, 0.2, 0.5, 0.6])")
print(f"u (correlated field): {np.round(u, 4)}  (expected: [0.7601, 0.7201, 0.8400, 0.8801])")
print()

# --- Phase 4 ---
k = {}
for (i, j) in edges:
    k[(i,j)] = np.exp((u[i] + u[j]) / 2)

print("=== Phase 4 ===")
for (i,j), k_e in k.items():
    print(f"  k_({i},{j}) = {k_e:.4f}")
print(f"  (expected: 2.0961, 2.1816, 2.3633, 2.2707)")
print()

# --- Phase 5 ---
L_k = lil_matrix((n_vertices, n_vertices))
for (i, j) in edges:
    k_e = k[(i,j)]
    L_k[i,i] += k_e
    L_k[j,j] += k_e
    L_k[i,j] -= k_e
    L_k[j,i] -= k_e
L_k = L_k.tocsr()

gamma_in  = [0]
gamma_out = [2]
p         = np.zeros(n_vertices)
p[0]      = 1.0
p[2]      = 0.0

boundary   = set(gamma_in) | set(gamma_out)
interior   = np.array([v for v in range(n_vertices) if v not in boundary])
L_interior = L_k[interior, :][:, interior]
rhs        = -L_k[interior, :][:, list(boundary)] @ p[list(boundary)]
p[interior] = spsolve(L_interior, rhs)

print("=== Phase 5 ===")
print(f"L_k:\n{np.round(L_k.toarray(), 4)}")
print(f"Pressure field p: {np.round(p, 4)}")
print(f"  (expected: p_0=1.0, p_1=0.4900, p_2=0.0, p_3=0.4899)")
print()

# --- Phase 6 ---
gamma_out_set = set(gamma_out)
gamma_in_set  = set(gamma_in)
Q    = 0.0
Q_in = 0.0

for (i, j) in edges:
    k_e = k.get((i,j), k.get((j,i), 1.0))
    if i in gamma_out_set or j in gamma_out_set:
        Q += k_e * abs(p[i] - p[j])
    if i in gamma_in_set or j in gamma_in_set:
        Q_in += k_e * abs(p[i] - p[j])

print("=== Phase 6 ===")
print(f"Q (flow out)    = {Q:.4f}  (expected: 2.2267)")
print(f"Q_in (flow in)  = {Q_in:.4f}  (expected: ~2.2267)")
print(f"Conservation holds: {np.isclose(Q_in, Q, atol=1e-4)}")

=== Phase 1 ===
A:
[[0. 1. 0. 1.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [1. 0. 1. 0.]]
D diagonal: [2. 2. 2. 2.]
L:
[[ 2. -1.  0. -1.]
 [-1.  2. -1.  0.]
 [ 0. -1.  2. -1.]
 [-1.  0. -1.  2.]]
L symmetric: True
L * 1 = [0. 0. 0. 0.] (should be all zeros)

=== Phase 2 ===
Eigenvalues of D^-1 L: [-0.  1.  1.  2.]
lambda_min = 1.0000
L_sigma:
[[ 2.5 -1.   0.  -1. ]
 [-1.   2.5 -1.   0. ]
 [ 0.  -1.   2.5 -1. ]
 [-1.   0.  -1.   2.5]]
Smallest eigenvalue of L_sigma: 0.5000 (should be > 0)

=== Phase 3 ===
w (edge noise)      : [ 0.5 -0.3  0.8 -0.2]
f (vertex sums)     : [0.3 0.2 0.5 0.6]  (expected: [0.3, 0.2, 0.5, 0.6])
u (correlated field): [0.76 0.72 0.84 0.88]  (expected: [0.7601, 0.7201, 0.8400, 0.8801])

=== Phase 4 ===
  k_(0,1) = 2.0959
  k_(1,2) = 2.1815
  k_(2,3) = 2.3632
  k_(3,0) = 2.2705
  (expected: 2.0961, 2.1816, 2.3633, 2.2707)

=== Phase 5 ===
L_k:
[[ 4.3664 -2.0959  0.     -2.2705]
 [-2.0959  4.2774 -2.1815  0.    ]
 [ 0.     -2.1815  4.5446 -2.3632]
 [-2.2705  0.     -2.3632  